# Utils for generation peptides

In [ ]:
from rdkit import Chem

peptide_smiles = {'A': 'C[C@H](N)C(=O)O',                        
                  'R': 'N=C(N)NCCC[C@H](N)C(=O)O',               
                  'N': 'NC(=O)C[C@H](N)C(=O)O',                  
                  'D': 'N[C@@H](CC(=O)O)C(=O)O',                 
                  'C': 'N[C@@H](CS)C(=O)O',                      
                  'Q': 'NC(=O)CC[C@H](N)C(=O)O',                 
                  'E': 'N[C@@H](CCC(=O)O)C(=O)O',                
                  'G': 'NCC(=O)O',                               
                  'H': 'N[C@@H](Cc1c[nH]cn1)C(=O)O',             
                  'I': 'CC[C@H](C)[C@H](N)C(=O)O',               
                  'L': 'CC(C)C[C@H](N)C(=O)O',                   
                  'K': 'NCCCC[C@H](N)C(=O)O',                   
                  'M': 'CSCC[C@H](N)C(=O)O',                     
                  'F': 'N[C@@H](Cc1ccccc1)C(=O)O',               
                  'P': 'O=C(O)[C@@H]1CCCN1',                     
                  'S': 'N[C@@H](CO)C(=O)O',                      
                  'T': 'C[C@@H](O)[C@H](N)C(=O)O',              
                  'W': 'N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O',      
                  'Y': 'N[C@@H](Cc1ccc(O)cc1)C(=O)O',          
                  'V': 'CC(C)[C@H](N)C(=O)O',                
                  'pS': 'N[C@@H](COP(=O)(O)O)C(=O)O',
                  'pT': 'P(O[C@H]([C@H](C(O)=O)N)C)(O)(O)=O',
                  'pY': 'C1C(C[C@H](N)C(O)=O)=CC=C(OP(O)(O)=O)C=1',
                  'pH1': 'C1N=CN(P(O)(O)=O)C=1C[C@@H](C(O)=O)N',
                  'pH2': 'N[C@@H](CC1N=CN(P(O)(O)=O)C=1)C(O)=O',
                  'sS': 'O(S(O)(=O)=O)C[C@@H](C(O)=O)N',
                  'sT': 'S(O[C@@H]([C@H](C(O)=O)N)C)(O)(=O)=O',
                  'sY': 'C1C(C[C@H](N)C(O)=O)=CC=C(OS(O)(=O)=O)C=1',
                  }

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

def condense_amino_acids(mol1, mol2):
    # define the reaction SMARTS for peptide bond formation
    rxn = AllChem.ReactionFromSmarts('[C:1](=[O:2])[OH].[N:3][C:4]>>[C:1](=[O:2])[N:3][C:4]')
    
    if mol1 is None or mol2 is None:
        return None
    
    # modify the molecules to ensure correct protonation states
    mol1 = Chem.AddHs(mol1)
    for atom in mol1.GetAtoms():
        if atom.GetSymbol() == 'O' and atom.GetFormalCharge() == -1:
            atom.SetFormalCharge(0)
    mol1 = Chem.RemoveHs(mol1)
    
    # modify the molecules to ensure correct protonation states
    mol2 = Chem.AddHs(mol2)
    for atom in mol2.GetAtoms():
        if atom.GetSymbol() == 'N' and atom.GetFormalCharge() == 1:
            atom.SetFormalCharge(0)
    mol2 = Chem.RemoveHs(mol2)
    
    # run the reaction
    products = rxn.RunReactants((mol1, mol2))
    
    if products:
        # get the first product
        product = products[0][0]
        
        # sanitize the product molecule
        Chem.SanitizeMol(product)
        
        return product
    else:
        return None

# example usage
peptide = condense_amino_acids(Chem.MolFromFASTA('A'), Chem.MolFromFASTA('S'))

if peptide:
    print(Chem.MolToSmiles(peptide, canonical=True))
else:
    print("failed to condense the amino acids")

C[C@H](N)C(=O)N[C@@H](CO)C(=O)O


In [3]:
import random
import numpy as np

# set random seed
random.seed(624)
np.random.seed(624)

def get_smiles_from_seq(seq):
    peptide = Chem.MolFromSmiles(peptide_smiles[seq[0]])
    for s in seq[1:]:
        peptide = condense_amino_acids(peptide, Chem.MolFromSmiles(peptide_smiles[s]))
    return peptide


# generate peptide with single site modified by Phosphorylation

In [ ]:
import math
from itertools import permutations
from tqdm import tqdm

peptide_mod_p = []
sequences_mod_p = []

elements = ['A', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'M', 'F', 'P', 'S', 'T', 'Y', 'V']
permutations_pep = list(permutations(elements, 3)) + list(permutations(elements, 4))


target_res_p = {'S', 'T', 'Y', 'H'}
for item in tqdm(permutations_pep, total=len(permutations_pep)):
    if not any(res in item for res in target_res_p):
        continue  

    # find all indices of target residues in the sequence
    target_indices = [i for i, res in enumerate(item) if res in target_res_p]

    # randomly select one index to replace
    replace_index = random.choice(target_indices)
    replaced_res = item[replace_index]

    
    # replace the selected residue with its phosphorylated form
    seq = list(item)  
    if replaced_res == 'H':
        rnd = random.random()
        if rnd >=0.5:
            seq[replace_index] = f'p{replaced_res}1'
        else:
            seq[replace_index] = f'p{replaced_res}2'
    else:
        seq[replace_index] = 'p' + replaced_res

    peptide = get_smiles_from_seq(seq)
    peptide_mod_p.append(Chem.MolToSmiles(peptide, canonical=True))
    sequences_mod_p.append(seq)

  0%|          | 0/61200 [00:00<?, ?it/s]

100%|██████████| 61200/61200 [01:42<00:00, 595.14it/s]


# generate peptide with single site modified by sulfation

In [ ]:
from itertools import permutations
from tqdm import tqdm
peptide_mod_s = []
sequences_mod_s = []

elements = ['A', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'M', 'F', 'P', 'S', 'T', 'Y', 'V']
permutations_pep = list(permutations(elements, 3)) + list(permutations(elements, 4))


target_res_p = {'S', 'T', 'Y'}
for item in tqdm(permutations_pep, total=len(permutations_pep)):
    if not any(res in item for res in target_res_p):
        continue  

    target_indices = [i for i, res in enumerate(item) if res in target_res_p]

    replace_index = random.choice(target_indices)
    replaced_res = item[replace_index]
    
    seq = list(item)
    seq[replace_index] = 's' + replaced_res

    peptide = get_smiles_from_seq(seq)
    peptide_mod_s.append(Chem.MolToSmiles(peptide, canonical=True))
    sequences_mod_s.append(seq)

100%|██████████| 61200/61200 [01:24<00:00, 725.73it/s] 


In [ ]:
from tqdm import tqdm
for i, smiles in tqdm(enumerate(peptide_mod_s), total=len(peptide_mod_s)):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    AllChem.MMFFOptimizeMolecule(mol)
    # conf = mol.GetConformer()
    Chem.MolToXYZFile(mol, f'./SI/peptide_s/peptide_s_xyz/{"-".join(sequences_mod_s[i])}.xyz')

 37%|███▋      | 13093/34992 [33:16<46:42,  7.81it/s]  

In [ ]:
from tqdm import tqdm
for i, smiles in tqdm(enumerate(peptide_mod_p), total=len(peptide_mod_p)):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    AllChem.MMFFOptimizeMolecule(mol)
    # conf = mol.GetConformer()
    Chem.MolToXYZFile(mol, f'./SI/peptide_3p/peptide_3p_xyz/{"-".join(sequences_mod_p[i])}.xyz')

100%|██████████| 42324/42324 [2:12:04<00:00,  5.34it/s]  
